# Stage 5: Data Preparation for Reporting

**Financial Forecasting & Variance Analysis** | Stage 5 of 6 | ← [Stage 4: Data Modelling & Analytics](../04-data-modelling-and-analytics/04-data-modelling-and-analytics.ipynb) | [Project README](../../README.md) | [Stage 6: Data Visualisation & Insights](../06-data-visualisation-and-insights/06-data-visualisation-and-insights.ipynb) →

> Reconcile the gold layer back to silver, then publish a small star-schema set of CSV extracts with a data dictionary for Power BI.

| Reads | Writes |
| --- | --- |
| `data/silver/*.parquet`, `data/gold/*.parquet` | `outputs/powerbi/*.csv` (gitignored), including `data_dictionary.csv` |

In [1]:
import sys
from pathlib import Path

# Locate the repo root (the folder that holds python/config.py) and import the shared modules
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "python" / "config.py").exists())
sys.path.insert(0, str(ROOT / "python"))

import pandas as pd
import config

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

import numpy as np
import warehouse

con = warehouse.connect()
q = lambda sql: warehouse.query(con, sql)

## 5.1 Reconciliation: gold back to silver

Every gold model must add back to the silver transactions it came from. These checks catch join fan-out (double counting) and dropped rows.

In [2]:
silver_total = q("SELECT SUM(amount) AS v, COUNT(*) AS n FROM silver_transactions").iloc[0]
complete_total = q("""
    SELECT SUM(t.amount) AS v FROM silver_transactions t
    JOIN gold_monthly_spend m ON m.month_start = CAST(date_trunc('month', t.transaction_date) AS DATE)
    WHERE m.is_complete_month""").iloc[0, 0]

checks = [
    ("Monthly model: complete-month net spend = silver complete-month total",
     q("SELECT SUM(net_spend) FROM gold_monthly_spend WHERE is_complete_month").iloc[0, 0], complete_total),
    ("Monthly model: all-month transactions = silver rows",
     q("SELECT SUM(transaction_count) FROM gold_monthly_spend").iloc[0, 0], silver_total["n"]),
    ("Customer profile: net spend = silver total",
     q("SELECT SUM(net_spend) FROM gold_customer_profile").iloc[0, 0], silver_total["v"]),
    ("Customer profile: transactions = silver rows",
     q("SELECT SUM(transaction_count) FROM gold_customer_profile").iloc[0, 0], silver_total["n"]),
    ("Category model: net spend = silver total",
     q("SELECT SUM(net_spend) FROM gold_category_monthly").iloc[0, 0], silver_total["v"]),
    ("Customer segments: one row per customer",
     q("SELECT COUNT(*) FROM gold_customer_segments").iloc[0, 0],
     q("SELECT COUNT(DISTINCT user_id) FROM silver_transactions").iloc[0, 0]),
]
recon = pd.DataFrame(checks, columns=["check", "gold", "silver"])
recon["difference"] = recon["gold"] - recon["silver"]
recon["status"] = np.where(np.isclose(recon["gold"], recon["silver"], rtol=1e-9, atol=0.01), "PASS", "FAIL")
recon

,check,gold,silver,difference,status
0,Monthly model: complete-month net spend = silv...,"571,835,522.28","571,835,522.28",-0.00,PASS
1,Monthly model: all-month transactions = silver...,"13,305,915.00","13,305,915.00",0.00,PASS
2,Customer profile: net spend = silver total,"571,835,522.28","571,835,522.28",-0.00,PASS
3,Customer profile: transactions = silver rows,"13,305,915.00","13,305,915.00",0.00,PASS
4,Category model: net spend = silver total,"571,835,522.28","571,835,522.28",-0.00,PASS
5,Customer segments: one row per customer,"1,219.00","1,219.00",0.00,PASS


In [3]:
if (recon["status"] == "FAIL").any():
    raise AssertionError("Gold does not reconcile to silver. Fix this before publishing extracts.")
print("Gold reconciles to silver. Safe to publish.")

Gold reconciles to silver. Safe to publish.


## 5.2 Reporting model (star schema)

| Extract | Type | Grain | Joins on |
| --- | --- | --- | --- |
| `dim_customer` | dimension | customer | `user_id` |
| `dim_month` | dimension | month | `month_start` |
| `dim_category` | dimension | MCC | `mcc_code` |
| `fact_monthly_portfolio` | fact | month × series (actual, backtest models, forward forecast) | `month_start` |
| `fact_customer_month_variance` | fact | customer × month | `user_id`, `month_start` |
| `fact_category_month` | fact | MCC × month | `mcc_code`, `month_start` |
| `fraud_profile`, `model_scores`, `variance_tests`, `data_quality_checks` | reference | (none) | (none) |

In [4]:
extracts = {}

extracts["dim_customer"] = q("""
    SELECT p.user_id, p.current_age, p.gender, p.yearly_income, p.total_debt, p.credit_score,
           p.cards_held, p.credit_cards, p.total_credit_limit, p.any_card_on_dark_web,
           p.first_transaction, p.last_transaction, p.tenure_months, p.top_category,
           s.segment, s.cluster, s.activity_status, s.R, s.F, s.M, s.RFM_score, s.value_rank,
           s.net_spend, s.avg_monthly_net_spend
    FROM gold_customer_profile p JOIN gold_customer_segments s USING (user_id)
""")

extracts["dim_month"] = q("""
    SELECT month_start, year, quarter, month, strftime(month_start, '%b %Y') AS month_label, is_complete_month
    FROM gold_monthly_spend
    UNION ALL
    SELECT CAST(month_start AS DATE), YEAR(month_start), QUARTER(month_start), MONTH(month_start),
           strftime(month_start, '%b %Y'), FALSE
    FROM gold_forecast_ahead
    ORDER BY month_start
""")

extracts["dim_category"] = q("SELECT mcc_code, category_description FROM silver_mcc_codes ORDER BY mcc_code")

actual = q("SELECT month_start, 'Actual' AS series, net_spend AS value, NULL AS lower, NULL AS upper "
           "FROM gold_monthly_spend WHERE is_complete_month")
backtest = (q("SELECT * FROM gold_forecast_backtest").drop(columns="actual")
            .melt(id_vars="month_start", var_name="series", value_name="value")
            .assign(series=lambda d: "Backtest: " + d["series"], lower=np.nan, upper=np.nan))
ahead = q("SELECT month_start, 'Forecast: SARIMA' AS series, forecast AS value, lower, upper FROM gold_forecast_ahead")
extracts["fact_monthly_portfolio"] = pd.concat([actual, backtest, ahead], ignore_index=True)

extracts["fact_customer_month_variance"] = q("""
    SELECT user_id, month_start, transaction_count, actual_spend, budget_spend, variance,
           variance_pct, variance_type, is_material
    FROM gold_customer_monthly_variance""")
extracts["fact_category_month"] = q("SELECT * EXCLUDE (category) FROM gold_category_monthly")
extracts["fraud_profile"] = q("SELECT * FROM gold_fraud_profile")
extracts["model_scores"] = q("SELECT * FROM gold_forecast_scores")
extracts["variance_tests"] = q("SELECT * FROM gold_variance_tests")
extracts["data_quality_checks"] = q("SELECT * FROM gold_data_quality_checks")

pd.DataFrame([{"extract": k, "rows": len(v), "columns": v.shape[1]} for k, v in extracts.items()])

,extract,rows,columns
0,dim_customer,1219,24
1,dim_month,130,6
2,dim_category,109,2
3,fact_monthly_portfolio,226,5
4,fact_customer_month_variance,136289,9
5,fact_category_month,12786,9
6,fraud_profile,40,6
7,model_scores,4,6
8,variance_tests,2,4
9,data_quality_checks,22,8


## 5.3 Publish extracts and the data dictionary

In [5]:
config.ensure_dirs()
dictionary = []
for name, df in extracts.items():
    df.to_csv(config.EXPORT_DIR / f"{name}.csv", index=False)
    dictionary += [{"extract": name, "column": c, "dtype": str(t),
                    "null_pct": round(df[c].isna().mean() * 100, 2)} for c, t in df.dtypes.items()]
dictionary = pd.DataFrame(dictionary)
dictionary.to_csv(config.EXPORT_DIR / "data_dictionary.csv", index=False)

pd.DataFrame([{"file": p.name, "size_mb": round(p.stat().st_size / 1e6, 2)}
              for p in sorted(config.EXPORT_DIR.glob("*.csv"))])

,file,size_mb
0,data_dictionary.csv,0.00
1,data_quality_checks.csv,0.00
2,dim_category.csv,0.00
3,dim_customer.csv,0.23
4,dim_month.csv,0.00
5,fact_category_month.csv,0.86
6,fact_customer_month_variance.csv,14.14
7,fact_monthly_portfolio.csv,0.01
8,fraud_profile.csv,0.00
9,model_scores.csv,0.00


## 5.4 Stage outcome

The reporting extracts reconcile to silver and are published to `outputs/powerbi/`, together with a column-level data dictionary. Transaction-level detail isn't exported: Power BI can read `data/silver/transactions.parquet` directly if drill-through is needed.

Next: [Stage 6: Data Visualisation & Insights](../06-data-visualisation-and-insights/06-data-visualisation-and-insights.ipynb)